# 02 — Train the MRZ line recognizer

One cell. Set `REPO`, pick a GPU pod, run.

**Run `01_synthetic_preview.ipynb` first.** Everything this model learns comes from the
generator; if the samples look wrong, the model will be wrong and this notebook will not
tell you.

## What this trains

A **PARSeq-lineage ViT encoder with a purpose-built fixed-length decoder** — not stock
PARSeq. The deviation is deliberate and worth understanding before you spend GPU hours:

| | stock PARSeq | here | why |
| --- | --- | --- | --- |
| input | 32×128 (4:1) | 32×704 (22:1) | 44 chars in 128px is 2.9px each — unreadable |
| encoder | ViT-Small, 12 layers | ViT-tiny, 6 layers | ViT-S measured **75.5ms/line** on CPU; two lines alone would blow the 100ms budget |
| decoder | autoregressive, 25 steps | one shot, 44 positions | MRZ has no language prior to learn, and Phase 4 wants per-position marginals |
| charset | 36 lowercase | 37 (`A-Z0-9<`) | the MRZ alphabet |
| loss | CE | CE + label smoothing | CTC solves unknown alignment and length; TD3 has neither |

**Output contract:** `(batch, 44, 37)` log-probs. Phase 4's beam search and ICAO validation
read exactly this.

## Expected cost

~38k steps at batch 128. This model is small and the generator is not: **the pod is
CPU-bound, not GPU-bound**, and wall-clock is set by how many cores render documents
rather than by which card you rent.

One core renders ~42 samples/s, so the ceiling is roughly `42 × cores` samples/s and
`it/s = samples/s ÷ batch_size`. Check it against `nvidia-smi`: a GPU sitting under 30%
means the generator is behind, and the levers are `num_workers` (defaults to one per
usable core), `batch_size`, and `dpi` — in that order.

In [ ]:
# ============================================================================
# Bootstrap: find the repo (or clone it), install what is missing.
# ============================================================================
import subprocess, sys, os, time, pathlib, importlib.util

REPO = "https://github.com/Kamisadev/mrz-ai.git"
WORKDIR = pathlib.Path("/workspace") if pathlib.Path("/workspace").exists() else pathlib.Path.cwd()

def sh(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

def refreshed(project):
    """Pull before training, or the pod trains whatever it had last time.

    A pod outlives a git push. Finding a checkout and using it as-is is how you
    spend an hour of GPU reproducing the exact model you were trying to replace,
    with nothing in the log to say so - the last run trained on one font that
    way. Fast-forward only: a pod with local edits should keep them and say so,
    not have them silently discarded.
    """
    if not (project / ".git").is_dir():
        return project
    try:
        sh("git", "-C", str(project), "pull", "--ff-only")
    except subprocess.CalledProcessError:
        print("WARNING: could not fast-forward. Training the code already on disk,")
        print("         which may not be what you just pushed.", flush=True)
    return project


def find_project():
    """Locate the repo root, whatever directory this notebook was opened from.

    Jupyter runs a notebook with the cwd set to the notebook's own folder, so
    checking only the cwd never matches when running from notebooks/ - it would
    silently clone a second copy and run that one instead.
    """
    for candidate in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (candidate / "src" / "mrz_ai").is_dir():
            return refreshed(candidate)
    for candidate in (WORKDIR / "mrz-ai", WORKDIR / "mrz_ai_v2"):
        if (candidate / "src" / "mrz_ai").is_dir():
            return refreshed(candidate)
    target = WORKDIR / "mrz-ai"
    sh("git", "clone", REPO, target)
    return target

PROJECT = find_project()
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))
print("project:", PROJECT)

# The real set lives outside git (docs/real.md), so it is here only if it was
# copied onto the pod. Nothing breaks without it -- the panel says so and the
# synthetic numbers carry on meaning exactly what they meant.
REAL = PROJECT / "real"
REAL = REAL if (REAL / "truth.json").is_file() else None
print("real set:", REAL or "none on this pod")

REQUIRED = {"PIL": "pillow", "numpy": "numpy", "cv2": "opencv-python-headless",
            "matplotlib": "matplotlib"}

REQUIRED["torch"] = "torch"
missing = [pkg for mod, pkg in REQUIRED.items() if not importlib.util.find_spec(mod)]
if missing:
    sh(sys.executable, "-m", "pip", "install", "-q", *missing)
else:
    print("dependencies already present")

import torch
from mrz_ai.training.recognition import TrainConfig, Stage, train_recognition

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

# Checkpoints go to the persistent volume so a stopped pod does not lose them.
OUTPUT = (WORKDIR / "checkpoints/recognition") if WORKDIR.name == "workspace" \
         else PROJECT / "checkpoints/recognition"

config = TrainConfig(
    batch_size=128,
    learning_rate=7e-4,
    # num_workers defaults to one per usable core, which is what keeps the GPU
    # fed: the generator is CPU-bound and the model is small enough that the
    # 3090 finishes a step long before the next batch is ready.
    output_dir=OUTPUT,
    # The real set, if this checkout has one. Specimen passports: real printing,
    # real typeface, invented identities -- read, never trained on. Absent by
    # default, and absent is fine; see docs/real.md.
    real_dir=REAL if REAL else None,
    # The blueprint's curriculum as a severity sweep. Each stage keeps the
    # easier range below it, so the model does not forget clean documents.
    curriculum=(
        Stage("clean",    (0.0, 0.05),  2_000),
        Stage("light",    (0.0, 0.35),  6_000),
        Stage("moderate", (0.0, 0.65), 10_000),
        Stage("heavy",    (0.0, 1.0),  20_000),
    ),
)

# ============================================================================
# The dashboard. Watch the run at http://<pod>:8080 -- expose the port in the
# pod's config first, or it is only reachable from inside the container.
#
# A separate process reading a file, not a thread inside training. It cannot
# slow the run down, cannot deadlock it, and cannot take it with it if it dies.
# ============================================================================
dashboard = subprocess.Popen(
    [sys.executable, "-m", "mrz_ai.serve.dashboard",
     "--dir", str(OUTPUT), "--port", "8080", "--host", "0.0.0.0"],
    cwd=str(PROJECT / "src"),
)
print("dashboard: http://localhost:8080  (pid", dashboard.pid, ")")

try:
    checkpoint = train_recognition(config)
    print("checkpoint:", checkpoint)
finally:
    # Left running, the pod keeps a port open on a run that has finished. The
    # last status is already on disk, so nothing is lost by stopping it.
    dashboard.terminate()

## Reading the numbers

`char` is per-character accuracy; `line` is the fraction of lines correct in *all 44*
positions. Watch `line` — one wrong character is a wrong document, so per-character
accuracy flatters the model badly here. At 44 characters, even 99.5% per character is only
~80% of lines correct.

Both are measured at full severity (0–1) on a disjoint generator seed, so there is no
leakage. Note what that does *not* mean: it is still synthetic evaluating synthetic. A
line accuracy of 99% here is a claim about this generator, not about passports.

Phase 4 will lift the effective accuracy above whatever this reports, because ICAO check
digits can reject a wrong hypothesis and promote the runner-up — but only for the fields
check digits cover. Sex and nationality have no protection at all (`docs/parser.md`).